# Giant-Donut Single-Sided Fit — Pupil Comparison

**Author:** Aaron Roodman
**Date Created:** 2026-06-14
**Last Modified:** 2026-06-15
**Status:** In Progress
**Keywords:** giant donut, FAM, defocus, danish, pupil, ISR, sky foreground, single-sided

## Description

Interactive tool to study **how well the optical pupil is modelled**, using the large
(8 mm) defocus FAM images (no donut tables / pipeline fits exist for these).

1. **Minimal ISR** on the raw (overscan + nominal gains only — no calibs).
2. **Sky foreground vs focal-plane radius** on the analyzed sensor (3σ-clipped mean per
   radius bin); optionally **subtract** this radial sky model before fitting, so the
   donut fit isn't fighting the sky gradient.
3. Display a CCD; **click a giant donut** → cut a ~1000 px stamp → **single-sided
   Danish fit**, default **pure-defocus Z4-only** with the **blur (fwhm) fixed at 1″**.
4. Show **data / model / residual**, a **Zernike bar chart**, the fitted **fwhm**, and
   a **4×N grid of slice profiles** (data vs model) across the donut.

`ipywidgets` controls change the **max Zernike term**, **binning**, **fix-fwhm**, and
**subtract-sky** live. Defocus from the `observation_reason` `intra_8mm`/`extra_8mm`
label (tunable).

**Output:** interactive in-notebook inspector (no files written by default).

**Based on:** `wfs_donut_selection_stages.ipynb`, `wfs_sky_foreground_radius.ipynb`;
raw in `LSSTCam/raw/all` (repo `/repo/main`).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-14 | Aaron Roodman | Initial version: minimal-ISR + click-to-select + single-sided Z4-only Danish fit for giant FAM donuts; data/model/residual pupil comparison |
| 2026-06-15 | Aaron Roodman | Add Zernike bar chart + fwhm text; widgets for max-Zernike term, binning, and fix-fwhm (default 1″); 4×N data/model slice-profile grid |
| 2026-06-15 | Aaron Roodman | Add sky-foreground vs focal-plane-radius (3σ-clipped mean) for the sensor, with a subtract-sky option before fitting |
| 2026-06-15 | Aaron Roodman | Fix title-string syntax (escaped-quote bug); default sensor R03_S12 (near R04 corner, ~518 e- sky gradient). Confirmed analytic jac=model.jac is used (njev>0) with the fwhm bounds injected via lstsqKwargs |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Sky Foreground vs Focal-Plane Radius](#sky)
6. [Interactive Giant-Donut Inspector](#interactive)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

butler_repo = "/repo/main"
raw_collection = "LSSTCam/raw/all"
instrument = "LSSTCam"
cam_name = "LSSTCam"

# Exposure: a labelled FAM defocus frame. observation_reason gives the defocus:
#   20251023: seq 337/338 = extra_8mm, 340/341 = intra_8mm (339 = unlabelled "test")
exposure = 2025102300340

# Detector. The pupil model is most interesting near the field CORNERS, so use a
# near-corner science sensor (clean defocus) or a WFS half-chip:
#   near-corner science: R03_S12, R01_S00, R10_S00, R30_S20, R34_S22, R41_S00, R43_S22
#   WFS half-chips:      R00_SW0/SW1, ...  (donut = defocus ± 1.5 mm)
detector_name = "R03_S12"

# ---- Fit defaults (also adjustable live via the widgets) ---------------
stamp_size = 1000                # postage-stamp size (px); giant donuts are ~450 px
binning = 4                      # danish binning (1=off/slow ~200s, 2/4 ~5s, 8 ~3s)
max_noll = 4                     # fit Noll 4..max_noll (4 = Z4-only / pure defocus)
start_with_intrinsic = False     # pure-defocus pupil model (no off-axis intrinsic terms)
fix_fwhm = True                  # fix the donut blur (fwhm) instead of fitting it
fwhm_arcsec = 1.0                # fixed fwhm value (arcsec) when fix_fwhm is True
subtract_sky = True              # subtract the radial sky model before fitting
defocus_mm = None                # None -> read from label; else override (e.g. 7.6)
defocal_type = None              # None -> from label ("intra"/"extra")
band = None                      # None -> from physical_filter (e.g. r_57 -> "r")

# ---- Sky foreground ----------------------------------------------------
sky_bin_mm = 2.0                 # focal-plane radius bin width (mm)
clip_sigma = 3.0                 # sigma for the clipped-mean sky estimate

# ---- Slices ------------------------------------------------------------
slice_width_px = 25              # band height per slice (fit-image px); ~20 slices at bin2
slice_ncols = 5                  # columns in the slice grid

# ---- Display ----------------------------------------------------------
cmap = "gray"
zernike_plot_range = None        # bar-chart ±µm; None -> auto-scale to the coefficients


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from astropy.stats import sigma_clipped_stats
from astropy.visualization import ZScaleInterval

import lsst.geom
from lsst.afw.cameraGeom import FIELD_ANGLE, PIXELS
from lsst.obs.lsst import LsstCam
from lsst.ip.isr import IsrTask, IsrTaskConfig
import lsst.daf.butler as dafButler
from galsim.zernike import noll_to_zern
from lsst.ts.wep import Image as WepImage
from lsst.ts.wep.estimation import WfEstimator
from lsst.ts.wep.utils import getTaskInstrument

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
from common.camera_utils import pixel_to_focal

setup_plotting()
warnings.filterwarnings("ignore")
camera = LsstCam.getCamera()
det_id = {d.getName(): d.getId() for d in camera}


<a id='functions'></a>
## Helper Functions

In [ ]:
def minimal_isr_config():
    """IsrTask config: overscan + assemble + nominal gains only (no calibs)."""
    cfg = IsrTaskConfig()
    for f in dir(cfg):
        if f.startswith("do") and isinstance(getattr(cfg, f), bool):
            try:
                setattr(cfg, f, False)
            except Exception:
                pass
    cfg.doOverscan = True
    cfg.doAssembleCcd = True
    cfg.doApplyGains = True
    cfg.doSaturation = True
    if hasattr(cfg, "usePtcGains"):
        cfg.usePtcGains = False
    return cfg


_ISR_TASK = None

def run_isr(butler, exposure, detector):
    """Overscan+gain ISR on one raw detector; returns the image array (electrons)."""
    global _ISR_TASK
    if _ISR_TASK is None:
        _ISR_TASK = IsrTask(config=minimal_isr_config())
    raw = butler.get("raw", collections=raw_collection,
                     dataId={"instrument": instrument, "exposure": exposure,
                             "detector": detector})
    return np.asarray(_ISR_TASK.run(raw, camera=camera).exposure.image.array, dtype=float)


def parse_defocus_label(reason):
    """Parse 'intra_8mm'/'extra_8mm' -> (defocal_type, defocus_mm) or (None, None)."""
    m = re.search(r"(intra|extra)_(\d+(?:\.\d+)?)mm", str(reason or ""))
    return (m.group(1), float(m.group(2))) if m else (None, None)


def focal_radius_map(detector_name, shape):
    """Per-pixel focal-plane radius (mm) for an assembled detector image."""
    ny, nx = shape
    xs = np.arange(nx, dtype=float); ys = np.arange(ny, dtype=float)
    xx, yy = np.meshgrid(xs, ys)
    fx, fy = pixel_to_focal(xx.ravel(), yy.ravel(), camera[detector_name])
    return np.hypot(fx, fy).reshape(shape).astype(np.float32)


def radial_sky_profile(img, rmap, bin_mm=2.0, sigma=3.0, min_pix=200):
    """3σ-clipped-mean sky in focal-plane radius bins -> (centers_mm, sky_e)."""
    edges = np.arange(float(rmap.min()), float(rmap.max()) + bin_mm, bin_mm)
    idx = np.digitize(rmap.ravel(), edges) - 1
    f = img.ravel()
    cen, sky = [], []
    for k in range(len(edges) - 1):
        sel = idx == k
        if int(sel.sum()) > min_pix:
            m, _, _ = sigma_clipped_stats(f[sel], sigma=sigma, maxiters=5)
            cen.append(0.5 * (edges[k] + edges[k + 1])); sky.append(m)
    return np.array(cen), np.array(sky)


def noll_list(max_noll):
    """Noll indices 4..max_noll (azimuthal pairs assumed complete for valid maxima)."""
    return list(range(4, int(max_noll) + 1))


def field_angle_deg(detector, x, y):
    """Field angle (deg) at pixel (x, y), from camera geometry."""
    fa = camera[detector].getTransform(PIXELS, FIELD_ANGLE).applyForward(
        lsst.geom.Point2D(x, y))
    return (np.degrees(fa.getX()), np.degrees(fa.getY()))


def single_sided_fit(stamp, field_angle, defocal_type, defocus_m, det_name, band,
                     noll, binning, start_intrinsic, fix_fwhm_value=None):
    """Single-sided Danish fit. If fix_fwhm_value is given, the blur (fwhm) is pinned
    there (via bounds passed through lstsqKwargs — the single-sided path otherwise
    leaves fwhm unbounded). Returns dict(data, model, residual, zk, fwhm, chi2) or None.

    Packed parameter vector is [flux, dx, dy, fwhm, bkg(1 for bkgOrder=0),
    z_fit(len noll)], so fwhm is index 3.
    """
    instr = getTaskInstrument(cam_name, det_name)
    instr.defocalOffset = defocus_m
    lstsq = {}
    if fix_fwhm_value is not None:
        n = 5 + len(noll)
        lo = [-np.inf] * n; hi = [np.inf] * n
        lo[0] = 0.0
        lo[3] = fix_fwhm_value - 1e-6; hi[3] = fix_fwhm_value + 1e-6
        lstsq["bounds"] = (lo, hi)
    wim = WepImage(image=stamp.copy(), fieldAngle=field_angle,
                   defocalType=defocal_type, bandLabel=band)
    wf = WfEstimator(algoName="danish",
                     algoConfig={"binning": binning, "lstsqKwargs": lstsq},
                     instConfig=instr, nollIndices=list(noll),
                     startWithIntrinsic=start_intrinsic, units="um", saveHistory=True)
    zk, meta = wf.estimateZk(wim, None)
    if zk is None or np.any(~np.isfinite(zk)):
        return None
    h = wf.history.get(defocal_type, {})
    if "image" not in h or "model" not in h or not np.all(np.isfinite(h["model"])):
        return None
    d = np.asarray(h["image"]); m = np.asarray(h["model"])
    return {"data": d, "model": m, "residual": d - m, "zk": np.asarray(zk),
            "fwhm": float(meta.get("fwhm", np.nan)),
            "chi2": float(meta.get("chi_square", np.nan))}


def zernike_bar(ax, zk_um, noll, plot_range=None):
    """Bar chart of Zernike coefficients (µm) vs Noll index, with radial-order bands."""
    ax.clear()
    noll = [int(j) for j in noll]; xs = np.arange(len(noll))
    ns = [noll_to_zern(j)[0] for j in noll]
    uniq = sorted(set(ns)); band = {nn: i for i, nn in enumerate(uniq)}
    i = 0
    while i < len(noll):
        j = i
        while j + 1 < len(noll) and ns[j + 1] == ns[i]:
            j += 1
        ax.axvspan(i - 0.5, j + 0.5, color=plt.cm.Pastel1(band[ns[i]] % 9),
                   alpha=0.35, zorder=0)
        i = j + 1
    ax.bar(xs, zk_um, width=0.6, color="k", zorder=2)
    ax.axhline(0, color="0.3", lw=0.8, zorder=1)
    pr = plot_range if plot_range else 1.1 * max(np.max(np.abs(zk_um)), 0.05)
    ax.set_ylim(-pr, pr); ax.set_xlim(-0.5, len(noll) - 0.5)
    ax.set_xticks(xs); ax.set_xticklabels([f"Z{j}" for j in noll], rotation=90, fontsize=7)
    ax.set_ylabel("µm")


def slice_profiles(data, model, slice_width):
    """List of (y0, y1, data_profile, model_profile) for horizontal bands."""
    h = data.shape[0]; n = h // slice_width
    return [(i * slice_width, (i + 1) * slice_width,
             np.nanmean(data[i * slice_width:(i + 1) * slice_width], axis=0),
             np.nanmean(model[i * slice_width:(i + 1) * slice_width], axis=0))
            for i in range(n)]


<a id='data'></a>
## Data Access

In [ ]:
butler = dafButler.Butler(butler_repo)
(exp_rec,) = butler.registry.queryDimensionRecords(
    "exposure", where=f"instrument='{instrument}' and exposure={exposure}")
lab_type, lab_mm = parse_defocus_label(exp_rec.observation_reason)
use_type = defocal_type or lab_type or "intra"
use_mm = defocus_mm or lab_mm or 8.0
use_band = band or str(exp_rec.physical_filter).split("_")[0]
print(f"exposure {exposure}: reason={exp_rec.observation_reason!r}, filter={exp_rec.physical_filter}")
print(f"  -> defocal_type={use_type}, defocus={use_mm} mm, band={use_band}")


<a id='sky'></a>
## Sky Foreground vs Focal-Plane Radius

In [ ]:
# ISR the analyzed sensor, then the 3-sigma-clipped sky vs focal-plane radius.
sensor_img = run_isr(butler, exposure, det_id[detector_name])
radius_map = focal_radius_map(detector_name, sensor_img.shape)
sky_centers, sky_vals = radial_sky_profile(sensor_img, radius_map, sky_bin_mm, clip_sigma)
sky_model_img = (np.interp(radius_map, sky_centers, sky_vals).astype(np.float32)
                 if len(sky_centers) else np.zeros_like(sensor_img))

print(f"{detector_name}: radius {radius_map.min():.1f}-{radius_map.max():.1f} mm, "
      f"sky {sky_vals.min():.1f}-{sky_vals.max():.1f} e- "
      f"(gradient {sky_vals.max()-sky_vals.min():.1f} e-)")

fig_sky, ax_sky = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax_sky.plot(sky_centers, sky_vals, "-o", ms=3)
ax_sky.set_xlabel("focal-plane radius [mm]")
ax_sky.set_ylabel(f"{clip_sigma:g}σ-clipped sky [e-]")
ax_sky.set_title(f"{detector_name} sky foreground vs focal-plane radius (exp {exposure})")
ax_sky.grid(alpha=0.3)
plt.show()


<a id='interactive'></a>
## Interactive Giant-Donut Inspector

Run the `%matplotlib widget` cell, then the controls + inspector cell. **Click a giant
donut** in the CCD: the tool fits it (on the sky-subtracted image, if enabled) and
updates data/model/residual, the Zernike bar chart, the fitted fwhm, and the 4×N slice
grid. The widgets re-fit the current donut live:

- **Max Zernike** — fit Noll 4..N (Z4 only = pure defocus, best for pupil comparison).
- **Binning** — danish binning (1 = off/slow; 2 → ~20 slices of 25 px; 4 = fast default).
- **Fix fwhm** + **fwhm (″)** — pin the blur (default 1″) so it can't absorb pupil mismatch.
- **Subtract sky** — subtract the radial sky model (above) before fitting.

In [ ]:
%matplotlib widget

In [ ]:
# Live controls
w_noll = widgets.Dropdown(
    options=[("Z4 only", 4), ("Z4–6", 6), ("Z4–11", 11), ("Z4–15", 15), ("Z4–22", 22)],
    value=max_noll, description="Max Zernike")
w_bin = widgets.Dropdown(options=[1, 2, 4, 8], value=binning, description="Binning")
w_fix = widgets.Checkbox(value=fix_fwhm, description="Fix fwhm")
w_fwhm = widgets.FloatText(value=fwhm_arcsec, description="fwhm (\")", step=0.1)
w_sky = widgets.Checkbox(value=subtract_sky, description="Subtract sky")
controls = widgets.HBox([w_noll, w_bin, w_fix, w_fwhm, w_sky])
display(controls)


In [ ]:
class GiantDonutInspector:
    """Click a giant donut -> single-sided Danish fit -> data/model/residual,
    Zernike bar chart, fwhm text, and a 4xN slice grid. Widgets re-fit live."""

    def __init__(self, detector_name, image, sky_model):
        self.det_name = detector_name
        self.det = det_id[detector_name]
        self.img = image
        self.sky_model = sky_model
        self.defocal_type = use_type
        self.defocus_m = use_mm * 1e-3
        self.band = use_band
        self.last = None
        z = ZScaleInterval().get_limits(self.img[np.isfinite(self.img)])

        self.fig = plt.figure(figsize=(12, 12))
        gs = self.fig.add_gridspec(3, 3, height_ratios=[2.4, 1, 1], hspace=0.25, wspace=0.3)
        self.ax_ccd = self.fig.add_subplot(gs[0, :])
        self.ax_st = {k: self.fig.add_subplot(gs[1, i])
                      for i, k in enumerate(("data", "model", "residual"))}
        self.ax_bar = self.fig.add_subplot(gs[2, :])
        self.ax_ccd.imshow(self.img, origin="lower", cmap=cmap, vmin=z[0], vmax=z[1],
                           interpolation="nearest")
        self.ax_ccd.set_title(f"{detector_name} (det {self.det}) exp {exposure} "
                              f"{self.defocal_type}_{use_mm:g}mm — click a donut",
                              fontsize=10)
        (self.marker,) = self.ax_ccd.plot([], [], "+", color="red", ms=16, mew=2)
        self.box = None
        for k, ax in self.ax_st.items():
            ax.set_title(k, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
        self.ax_bar.set_title("Zernike coefficients — click a donut", fontsize=9)

        self.fig_sl, self.ax_sl = plt.subplots(
            1, slice_ncols, figsize=(2.4 * slice_ncols, 2.2),
            constrained_layout=True, squeeze=False)
        self.fig_sl.suptitle("data (black) vs model (red) — row slices", fontsize=10)

        self.cid = self.fig.canvas.mpl_connect("button_press_event", self.on_click)
        for w in (w_noll, w_bin, w_fix, w_fwhm, w_sky):
            w.observe(self._on_widget, names="value")

    def _image_for_fit(self):
        return self.img - self.sky_model if w_sky.value else self.img

    def _on_widget(self, _change):
        if self.last is not None:
            self._fit_and_draw(*self.last)

    def on_click(self, event):
        if event.inaxes is not self.ax_ccd or event.xdata is None:
            return
        self._fit_and_draw(int(round(event.xdata)), int(round(event.ydata)))

    def _fit_and_draw(self, cx, cy):
        half = stamp_size // 2
        ny, nx = self.img.shape
        if not (half <= cx < nx - half and half <= cy < ny - half):
            print("too close to the edge for a full stamp"); return
        self.last = (cx, cy)
        stamp = self._image_for_fit()[cy - half:cy + half, cx - half:cx + half].astype(float)
        self.marker.set_data([cx], [cy])
        if self.box is not None:
            self.box.remove()
        self.box = plt.Rectangle((cx - half, cy - half), stamp_size, stamp_size,
                                 fill=False, ec="red", lw=1.2)
        self.ax_ccd.add_patch(self.box)

        noll = noll_list(w_noll.value)
        fa = field_angle_deg(self.det, cx, cy)
        res = single_sided_fit(stamp, fa, self.defocal_type, self.defocus_m,
                               self.det_name, self.band, noll, w_bin.value,
                               start_with_intrinsic,
                               fix_fwhm_value=(w_fwhm.value if w_fix.value else None))
        for k in ("data", "model", "residual"):
            self.ax_st[k].clear(); self.ax_st[k].set_xticks([]); self.ax_st[k].set_yticks([])
        self.ax_bar.clear()
        for ax in self.ax_sl.flat:
            ax.clear(); ax.set_xticks([]); ax.set_yticks([])

        if res is None:
            self.ax_st["model"].set_title("fit failed (cleaner/centred donut?)", fontsize=9)
            print(f"({cx},{cy}): fit failed")
        else:
            for k, cm in (("data", "viridis"), ("model", "viridis"),
                          ("residual", "RdBu_r")):
                arr = res[k]; kw = {}
                if k == "residual":
                    v = np.nanpercentile(np.abs(arr), 99); kw = dict(vmin=-v, vmax=v)
                self.ax_st[k].imshow(arr, origin="lower", cmap=cm, **kw)
                self.ax_st[k].set_title(k, fontsize=9)
                self.ax_st[k].set_xticks([]); self.ax_st[k].set_yticks([])
            zernike_bar(self.ax_bar, res["zk"], noll, zernike_plot_range)
            self.ax_bar.set_title("Zernike coefficients (µm)", fontsize=9)
            self._draw_slices(res["data"], res["model"])
            rr = np.std(res["residual"]) / np.std(res["data"])
            fwhm_tag = "fix" if w_fix.value else "fit"
            sky_tag = "-sub" if w_sky.value else ""
            self.ax_ccd.set_title(
                f"{self.det_name} exp {exposure} {self.defocal_type}_{use_mm:g}mm  "
                f"({cx},{cy})  fwhm={res['fwhm']:.2f}″ {fwhm_tag}  "
                f"sky{sky_tag}  Z4={res['zk'][0]*1e3:.0f} nm  "
                f"resid/data={rr:.2f}", fontsize=9)
            print(f"({cx},{cy}) noll=4..{noll[-1]} bin={w_bin.value} sky_sub={w_sky.value} "
                  f"fwhm={res['fwhm']:.3f}″ Z4={res['zk'][0]*1e3:.0f}nm resid/data={rr:.3f}")
        self.fig.canvas.draw_idle()
        self.fig_sl.canvas.draw_idle()

    def _draw_slices(self, data, model):
        slices = slice_profiles(data, model, slice_width_px)
        n = len(slices); nrows = max(1, int(np.ceil(n / slice_ncols)))
        if self.ax_sl.shape != (nrows, slice_ncols):
            self.fig_sl.clear()
            self.ax_sl = self.fig_sl.subplots(nrows, slice_ncols, squeeze=False)
            self.fig_sl.suptitle("data (black) vs model (red) — row slices", fontsize=10)
        self.fig_sl.set_size_inches(2.4 * slice_ncols, 1.9 * nrows)
        for i, ax in enumerate(self.ax_sl.flat):
            ax.clear()
            if i < n:
                y0, y1, dp, mp = slices[i]
                ax.plot(dp, "k.", ms=1.5); ax.plot(mp, "r-", lw=1.0)
                ax.set_title(f"{y0}-{y1}", fontsize=6); ax.tick_params(labelsize=5)
            else:
                ax.axis("off")


inspector = GiantDonutInspector(detector_name, sensor_img, sky_model_img)
